In [ ]:

import pandas as pd
import numpy as np
import re

from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# 1. Load dataset
data = pd.read_csv('IMDB Dataset.csv')

# 2. Preprocess text
stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = re.sub('[^a-zA-Z]', ' ', str(text))
    words = text.lower().split()
    words = [w for w in words if w not in stop_words]
    return words

sentences = data['review'].apply(clean_text).tolist()

# Convert labels into numbers
y = data['sentiment'].map({'positive': 1, 'negative': 0}).values

# 3. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    sentences, y, test_size=0.2, random_state=42
)

# 4. Train CBOW Word2Vec model
w2v = Word2Vec(
    sentences=X_train,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,
    workers=4
)

# 5. Convert documents into vectors
def document_vector(words):
    vectors = [w2v.wv[word] for word in words if word in w2v.wv]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

X_train_vec = np.array([document_vector(doc) for doc in X_train])
X_test_vec = np.array([document_vector(doc) for doc in X_test])

# 6. Build classification model
model = Sequential([
    Dense(64, activation='relu', input_shape=(100,)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

# 7. Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 8. Train model
history = model.fit(
    X_train_vec, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

# 9. Evaluate model
loss, accuracy = model.evaluate(X_test_vec, y_test)

print("Test Accuracy:", accuracy)
